# Đánh giá cuối MobileNetV3 và YOLOv8n-cls trên Colab

Notebook này chỉ chạy inference. Nó xác minh hai run dùng cùng dataset/code, chọn threshold và model thắng **chỉ trên validation**, ghi quyết định bất biến, rồi mới mở test set. Kết quả gồm accuracy, precision/recall/F1, damage recall, false reject rate, confusion matrix và prediction table có SHA-256. Phạm vi hiện tại là model nhị phân v0 `damage/normal`, nên kết quả được gắn `smoke_only`.

In [ ]:
REPO_URL = "https://github.com/DangHung1110/IoT-CK.git"
GIT_REF = "feat/model-training-pipeline"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/parcel_sorter"  # @param {type:"string"}
MOBILENET_RUN_ID = "20261103t153037z-mobilenet_v3_small-v0-s42-7c300430"  # @param {type:"string"}
YOLO_RUN_ID = "20261103t153335z-yolov8n_cls-v0-s42-7c300430"  # @param {type:"string"}
THRESHOLD_GRID = "0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90,0.95"
MIN_DAMAGE_SAFETY_RECALL = 0.95  # @param {type:"number"}
MAX_FALSE_REJECT_RATE = 0.10  # @param {type:"number"}

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import re
import subprocess
from pathlib import Path

repo_dir = Path("/content/IoT-CK")
if not (repo_dir / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(repo_dir)],
        check=True,
    )
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", "--prune"], check=True)
target_ref = GIT_REF if re.fullmatch(r"[0-9a-f]{40}", GIT_REF) else f"origin/{GIT_REF}"
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", target_ref], check=True)

In [ ]:
import importlib
import sys

if not ((3, 11) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError(f"Unsupported Colab Python: {sys.version.split()[0]}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir), "--no-deps"],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-r",
        str(repo_dir / "requirements" / "colab-training-overlay.txt"),
    ],
    check=True,
)
src_dir = str(repo_dir / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
for module_name in tuple(sys.modules):
    if module_name == "parcel_sorter" or module_name.startswith("parcel_sorter."):
        del sys.modules[module_name]
importlib.invalidate_caches()
parcel_sorter = importlib.import_module("parcel_sorter")
package_path = Path(parcel_sorter.__file__).resolve()
if not package_path.is_relative_to(repo_dir.resolve()):
    raise RuntimeError(f"parcel_sorter loaded from unexpected path: {package_path}")
print(f"Loaded parcel_sorter from {package_path}")

In [ ]:
import shutil
from datetime import UTC, datetime

from parcel_sorter.colab import validate_colab_drive_root
from parcel_sorter.data import DatasetSplit, sha256_file
from parcel_sorter.ml.colab_data import stage_verified_dataset
from parcel_sorter.ml.colab_evaluation import (
    resolve_best_checkpoint,
    validate_binary_run_pair,
)
from parcel_sorter.ml.contracts import ModelCandidate
from parcel_sorter.ml.run_metadata import load_training_run

drive_root = validate_colab_drive_root(DRIVE_ROOT)
runs_root = drive_root / "models" / "runs"
run_dirs = {
    ModelCandidate.MOBILENET_V3_SMALL: runs_root / MOBILENET_RUN_ID,
    ModelCandidate.YOLOV8N_CLS: runs_root / YOLO_RUN_ID,
}
run_manifests = tuple(load_training_run(path) for path in run_dirs.values())
runs = validate_binary_run_pair(run_manifests)
reference = runs[ModelCandidate.MOBILENET_V3_SMALL]
local_workspace = Path("/content/parcel_sorter_eval")
if local_workspace.exists():
    shutil.rmtree(local_workspace)
prepared = stage_verified_dataset(
    archive_source=reference.dataset.archive_source,
    manifest_source=reference.dataset.manifest_source,
    expected_archive_sha256=reference.dataset.archive_sha256,
    expected_manifest_sha256=reference.dataset.manifest_sha256,
    drive_root=drive_root,
    local_workspace=local_workspace,
    class_order=reference.dataset.class_order,
)
checkpoints = {
    candidate: resolve_best_checkpoint(run_dirs[candidate], candidate)
    for candidate in ModelCandidate
}
checkpoint_hashes = {candidate: sha256_file(path) for candidate, path in checkpoints.items()}
evaluation_id = (
    datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ").lower() + f"-binary-v0-{reference.git_commit[:8]}"
)
evaluation_dir = drive_root / "models" / "evaluations" / evaluation_id
evaluation_dir.mkdir(parents=True, exist_ok=False)
threshold_grid = tuple(float(item) for item in THRESHOLD_GRID.split(","))
print(f"Evaluation output: {evaluation_dir}")
print({candidate.value: checkpoint_hashes[candidate] for candidate in ModelCandidate})

In [ ]:
from parcel_sorter.ml.binary_evaluation import (
    BinaryCandidateValidationResult,
    evaluate_binary_predictions,
    save_binary_prediction_table,
    select_binary_model_candidate,
    select_binary_validation_thresholds,
)
from parcel_sorter.ml.colab_evaluation import predict_binary_candidate
from parcel_sorter.ml.training_runtime import write_json_atomic

validation_results = {}
validation_prediction_hashes = {}
for candidate in ModelCandidate:
    records = predict_binary_candidate(
        config=runs[candidate].config,
        prepared=prepared,
        checkpoint=checkpoints[candidate],
        split=DatasetSplit.VAL,
    )
    candidate_dir = evaluation_dir / candidate.value
    prediction_path = candidate_dir / "predictions_val.jsonl"
    validation_prediction_hashes[candidate] = save_binary_prediction_table(prediction_path, records)
    selection = select_binary_validation_thresholds(
        records,
        threshold_grid,
        min_damage_safety_recall=MIN_DAMAGE_SAFETY_RECALL,
        max_false_reject_rate=MAX_FALSE_REJECT_RATE,
    )
    report = evaluate_binary_predictions(
        records, selection.best.thresholds, required_split=DatasetSplit.VAL
    )
    result = BinaryCandidateValidationResult(
        candidate=candidate,
        checkpoint_sha256=checkpoint_hashes[candidate],
        thresholds=selection.best.thresholds,
        constraint_met=selection.best.constraint_met,
        report=report,
    )
    validation_results[candidate] = result
    write_json_atomic(candidate_dir / "threshold_sweep.json", selection.model_dump(mode="json"))
    write_json_atomic(candidate_dir / "validation_report.json", result.model_dump(mode="json"))

winner = select_binary_model_candidate(tuple(validation_results.values()))
frozen_selection_path = evaluation_dir / "selection_frozen.json"
write_json_atomic(
    frozen_selection_path,
    {
        "selection_basis": "validation_only",
        "winner": winner.candidate.value,
        "winner_checkpoint_sha256": winner.checkpoint_sha256,
        "winner_thresholds": winner.thresholds.model_dump(mode="json"),
        "dataset_archive_sha256": reference.dataset.archive_sha256,
        "dataset_manifest_sha256": reference.dataset.manifest_sha256,
        "validation_prediction_sha256": {
            candidate.value: digest for candidate, digest in validation_prediction_hashes.items()
        },
        "candidates": {
            candidate.value: result.model_dump(mode="json")
            for candidate, result in validation_results.items()
        },
    },
)
print(f"Winner frozen before test: {winner.candidate.value}")
print(winner.thresholds.model_dump())

In [ ]:
if not frozen_selection_path.is_file():
    raise RuntimeError("Selection must be frozen before test evaluation")

test_reports = {}
test_prediction_hashes = {}
for candidate in ModelCandidate:
    records = predict_binary_candidate(
        config=runs[candidate].config,
        prepared=prepared,
        checkpoint=checkpoints[candidate],
        split=DatasetSplit.TEST,
    )
    candidate_dir = evaluation_dir / candidate.value
    prediction_path = candidate_dir / "predictions_test.jsonl"
    test_prediction_hashes[candidate] = save_binary_prediction_table(prediction_path, records)
    report = evaluate_binary_predictions(
        records,
        validation_results[candidate].thresholds,
        required_split=DatasetSplit.TEST,
    )
    test_reports[candidate] = report
    write_json_atomic(candidate_dir / "test_report.json", report.model_dump(mode="json"))

comparison = {
    "quality_status": "smoke_only",
    "selection_basis": "validation_only",
    "winner": winner.candidate.value,
    "winner_thresholds": winner.thresholds.model_dump(mode="json"),
    "test_did_not_change_winner": True,
    "candidates": {
        candidate.value: {
            "checkpoint_sha256": checkpoint_hashes[candidate],
            "validation": validation_results[candidate].report.model_dump(mode="json"),
            "test": test_reports[candidate].model_dump(mode="json"),
            "test_prediction_sha256": test_prediction_hashes[candidate],
        }
        for candidate in ModelCandidate
    },
}
write_json_atomic(evaluation_dir / "comparison.json", comparison)

In [ ]:
import matplotlib.pyplot as plt

labels = [item.value for item in reference.config.classes]
figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for axis, candidate in zip(axes, ModelCandidate, strict=True):
    report = test_reports[candidate]
    matrix = [
        [report.confusion_matrix[actual][predicted] for predicted in labels] for actual in labels
    ]
    image = axis.imshow(matrix, cmap="Blues")
    axis.set_title(f"{candidate.value} - test")
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")
    axis.set_xticks(range(len(labels)), labels)
    axis.set_yticks(range(len(labels)), labels)
    for row in range(len(labels)):
        for column in range(len(labels)):
            axis.text(column, row, matrix[row][column], ha="center", va="center")
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
figure.tight_layout()
confusion_path = evaluation_dir / "test_confusion_matrices.png"
figure.savefig(confusion_path, dpi=180, bbox_inches="tight")
plt.show()

header = (
    f"{'candidate':<20} {'acc':>7} {'damage_R':>10} {'damage_F1':>10} "
    f"{'false_rej':>10} {'macro_F1':>10}"
)
print(header)
for candidate in ModelCandidate:
    report = test_reports[candidate]
    print(
        f"{candidate.value:<20} {report.accuracy:>7.3f} "
        f"{report.damage_class_recall:>10.3f} {report.damage_class_f1:>10.3f} "
        f"{report.false_reject_rate:>10.3f} {report.macro_f1:>10.3f}"
    )
print(f"\nFrozen winner: {winner.candidate.value}")
print(f"Thresholds: {winner.thresholds.model_dump()}")
print(f"Reports saved to: {evaluation_dir}")
print(f"Confusion matrix: {confusion_path}")